In [2]:
import torch
import torch.nn as nn


avg_pool = nn.AvgPool1d(kernel_size=8, stride=8, )

input_tensor = torch.randn(2, 128, 25000) # (batch_size, channels, width)

input_tensor = input_tensor.transpose(1, 2)  # Now shape is (batch_size, width, channels)

# Apply average pooling
output = avg_pool(input_tensor)

# Transpose back to original dimensions
output = output.transpose(1, 2)  
print(output.size()) # torch.Size([2, 128, 3125])



torch.Size([2, 16, 25000])


In [2]:
model.config

GPT2Config {
  "_name_or_path": "gpt2",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.41.2",
  "use_cache": true,
  "vocab_size": 50257
}

In [6]:
from sae_lens import SAE
import yaml
from transformers import GPT2Model, GPT2ForSequenceClassification, GPT2Tokenizer

import numpy as np
from datasets import load_dataset

In [4]:
dataset = load_dataset('imdb')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

/root/sae/v/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
token_lengths = []

for i in range(len(dataset['train'])):
    token_lengths.append(len(tokenizer(dataset['train'][i]['text'])['input_ids']))
for i in range(len(dataset['test'])):
    token_lengths.append(len(tokenizer(dataset['test'][i]['text'])['input_ids']))

Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


In [7]:
# print the 99th, 95th, 90th, 75th, 50th, 25th, 10th, 5th, 1st percentiles

percentiles = [99, 95, 90, 75, 50, 25, 10, 5, 1]

for p in percentiles:
    print(f'{p}th percentile: {np.percentile(token_lengths, p)}')


99th percentile: 1158.0
95th percentile: 763.0
90th percentile: 583.0
75th percentile: 360.0
50th percentile: 220.0
25th percentile: 159.0
10th percentile: 115.0
5th percentile: 79.0
1th percentile: 52.0


In [11]:
np.random.seed(33)

print(np.random.rand(2))

[0.24851013 0.44997542]


In [4]:
# show all parameters in the sae

for name, param in sae.named_parameters():
    print(name, param.size())

b_enc torch.Size([24576])
W_dec torch.Size([24576, 768])
W_enc torch.Size([768, 24576])
b_dec torch.Size([768])


In [2]:
with open('../app/assets/pretrained_saes.yaml', 'r') as f:
    pretrained_saes = yaml.safe_load(f)

# for k, v in pretrained_saes['SAE_LOOKUP'].items():
#     print(k)
#     print(v.keys())

#     for sae in v['saes']:
#         print(sae['id'])


In [5]:
for k, v in pretrained_saes['SAE_LOOKUP'].items():

    if k == 'gpt2-small-res-jb':
        continue

    for sae in v['saes']:
        print(k, sae['id'])




        sae, _, _ = SAE.from_pretrained(
            release = k, # see other options in sae_lens/pretrained_saes.yaml
            sae_id = sae['id'], # won't always be a hook point
            device = "cpu"
        )

        print(sae.cfg.context_size)

gpt2-small-hook-z-kk blocks.0.hook_z
{'d_in': 768, 'd_sae': 24576, 'dtype': 'float32', 'device': 'cpu', 'model_name': 'gpt2-small', 'hook_name': 'blocks.0.attn.hook_z', 'hook_layer': 0, 'hook_head_index': None, 'activation_fn_str': 'relu', 'apply_b_dec_to_input': True, 'finetuning_scaling_factor': False, 'sae_lens_training_version': None, 'prepend_bos': True, 'dataset_path': 'apollo-research/Skylion007-openwebtext-tokenizer-gpt2', 'context_size': 128, 'normalize_activations': 'none'}
<class 'sae_lens.sae.SAEConfig'>


TypeError: SAEConfig.__init__() missing 1 required positional argument: 'dataset_trust_remote_code'

In [ ]:
def classify_sentiment(model, batch):
    with torch.no_grad():
        outputs = model(input_ids=batch)
    logits = outputs.logits[:, -1, :]
    probabilities = softmax(logits, dim=-1)
    probabilities = probabilities[:, TOKENIZED_LABELS]
    predicted_idxs = torch.argmax(probabilities, dim=-1)
    predicted_sentiments = [LABELS[idx.item()] for idx in predicted_idxs]
    return predicted_sentiments

def classify_sentiment_generate(model, batch):
    outputs = model.generate(input_ids=batch, max_length=batch.shape[1] + 1, num_return_sequences=1)
    decoded_outputs = [tokenizer.decode(output) for output in outputs]
    predicted_sentiments = []
    for decoded_output in decoded_outputs:
        last_word = decoded_output.split()[-1].lower()
        if last_word in LABELS:
            predicted_sentiments.append(last_word)
        else:
            predicted_sentiments.append(classify_sentiment_random(model, [decoded_output])[0])
    return predicted_sentiments

def classify_sentiment_random(model, batch):
    return ['positive' if torch.rand(1) < 0.5 else 'negative' for _ in batch]

In [ ]:
sae.cfg.context_size

128

In [4]:
from sae_lens import SAE


In [5]:
sae, _, _ = SAE.from_pretrained(
    release = "mistral-7b-res-wg", # see other options in sae_lens/pretrained_saes.yaml
    sae_id = "blocks.8.hook_resid_pre", # won't always be a hook point
    device = "cuda"
)   

In [6]:
sae.cfg

SAEConfig(d_in=4096, d_sae=65536, activation_fn_str='relu', apply_b_dec_to_input=False, finetuning_scaling_factor=False, context_size=256, model_name='mistral-7b', hook_name='blocks.8.hook_resid_pre', hook_layer=8, hook_head_index=None, prepend_bos=False, dataset_path='monology/pile-uncopyrighted', dataset_trust_remote_code=True, normalize_activations='constant_norm_rescale', dtype='float32', device='cuda', sae_lens_training_version=None)

In [ ]:
7

In [ ]:
from transformer_lens import HookedTransformer
import json

with open('../.credentials.json') as f:
    creds = json.load(f)


import os
os.environ['HF_TOKEN']=creds['HF_TOKEN']

model = HookedTransformer.from_pretrained('mistralai/Mistral-7B-v0.1', device='cuda', use_auth_token=creds['HF_TOKEN'])

/root/sae/v/lib/python3.10/site-packages/transformers/models/auto/configuration_auto.py:919: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/root/sae/v/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/root/sae/v/lib/python3.10/site-packages/transformers/models/auto/auto_factory.py:468: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards: 100%|██████████| 2/2 [00:43<00:00, 21.69s/it]


OSError: Can't load tokenizer for 'mistralai/Mistral-7B-v0.1'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'mistralai/Mistral-7B-v0.1' is the correct path to a directory containing all relevant files for a LlamaTokenizerFast tokenizer.

In [ ]:
from huggingface_hub import notebook_login
# log in to huggingfacenotebook_login()
notebook_login()


In [15]:
!pip install ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 KB 3.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.4/214.4 KB 24.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 42.5 MB/s eta 0:00:0000:01
